## Extension Analysis Work (on-going)

In [3]:
from tabulate import tabulate
import pandas as pd

In [4]:
def joules_to_kwh(joules):
    return float(joules) / 3600000

def load_philipp_data(cluster, workflow, runs):
    processed = {}

    for run in range(1, runs+1):
        with open(f'nxf-experiments/{cluster}-cluster/{workflow}/{run}/wf_data.csv') as file:
            stripped_lines = [line.rstrip().split(',') for line in file.readlines()]
            data = stripped_lines[1:]

        pkg = data[0][11]
        dram = data[0][12]
        total = data[0][13]

        processed[run] = {}
        processed[run]['pkg'] = joules_to_kwh(pkg)
        processed[run]['dram'] = joules_to_kwh(dram)
        processed[run]['total'] = joules_to_kwh(total)
    
    return processed


def load_data_totals(cluster, workflow, runs):
    processed = {}

    for run in range(1, runs+1):
        proc_data = pd.read_csv(f'nxf-experiments/{cluster}-cluster/{workflow}/{run}/trace.csv', header=0)
        processed[run] = {}
        processed[run]['total_read_bytes'] = proc_data['read_bytes'].sum()
        processed[run]['total_write_bytes'] = proc_data['write_bytes'].sum()

    return processed


def load_philipp_total_energy(cluster, workflow, runs):
    processed = {}
        
    for run in range(1, runs+1):
        with open(f'nxf-experiments/{cluster}-cluster/{workflow}/{run}/task_data.md') as file:
            lines = [line.strip() for line in file.readlines()]

        final = lines[-1].split(': ')[1][:-6]

        processed[run] = {}
        processed[run]['pkg'] = None
        processed[run]['dram'] = None
        processed[run]['total'] = joules_to_kwh(final)
    
    return processed

In [5]:
# Functions to parse rapl data
def load_workflow_rapl_readings(cluster, workflow):
    with open(f'nxf-experiments/{cluster}-cluster/{workflow}-runs.csv', 'r') as file:
        data = [line.strip().split(',') for line in file.readlines()]
    
    return data[1:]

def process_rapl_data(data):
    processed = {}
    run = 1

    for row in data:
        processed[run] = {}
        processed[run]['pkg'] = float(row[1]) - float(row[4])
        processed[run]['dram'] = float(row[2]) - float(row[5])
        processed[run]['total'] = float(row[3]) - float(row[6])
        processed[run]['pkg_active'] = float(row[7])
        processed[run]['dram_active'] = float(row[8])
        processed[run]['total_active'] = float(row[9])
        run += 1

    return processed

In [6]:
# Functions to parse ichnos data
def load_ichnos_summary_file(summary_file):
    with open(summary_file, 'r') as file:
        raw = [line.strip() for line in file.readlines()]

    cpu = float(raw[7].split(':')[1].strip()[:-3])
    task_mem = float(raw[9].split(':')[1].strip()[:-3])
    node_mem = float(raw[15].split(':')[1].strip()[:-3])

    return (cpu, task_mem, node_mem)

def load_ichnos_data(cluster, workflow, runs, strategy):
    workflow_path = f'nxf-experiments/{cluster}-cluster/{workflow}'
    data = {}

    for run in range(1, runs + 1):
        data[run] = {}
        (cpu, task_mem, node_mem) = load_ichnos_summary_file(f'{workflow_path}/{run}/{cluster}-{workflow}-{run}-1-{strategy}-summary.txt')
        data[run]['pkg'] = cpu
        data[run]['dram'] = task_mem + node_mem 
        data[run]['total'] = cpu + task_mem + node_mem

    return data

In [7]:
# Generic functions
def get_error(experimental, actual):
    return round((abs(experimental - actual) / actual) * 100, 2)

In [8]:
print('Cluster Workflow Executions')

rapl_data = {'hu': {}, 'gu': {}}
rapl_data['hu']['rnaseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'rnaseq'))
rapl_data['hu']['chipseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'chipseq'))
# rapl_data['gu']['rnaseq'] = process_rapl_data(load_workflow_rapl_readings('gu', 'rnaseq'))
rapl_data['hu']['sarek'] = load_philipp_total_energy('hu', 'sarek', 2)

ichnos_data = {'hu': {}, 'gu': {}}
ichnos_data['hu']['rnaseq'] = load_ichnos_data('hu', 'rnaseq', len(rapl_data['hu']['rnaseq']), 'schedutil_linear')
ichnos_data['hu']['chipseq'] = load_ichnos_data('hu', 'chipseq', len(rapl_data['hu']['chipseq']), 'schedutil_linear')
# ichnos_data['gu']['rnaseq'] = load_ichnos_data('gu', 'rnaseq', len(rapl_data['gu']['rnaseq']), 'ondemand_linear')
ichnos_data['hu']['sarek'] = load_ichnos_data('hu', 'sarek', len(rapl_data['hu']['sarek']), 'schedutil_linear')

bytes_data = {'hu': {}, 'gu': {}}
bytes_data['hu']['rnaseq'] = load_data_totals('hu', 'rnaseq', len(rapl_data['hu']['rnaseq']))
bytes_data['hu']['chipseq'] = load_data_totals('hu', 'chipseq', len(rapl_data['hu']['chipseq']))
bytes_data['hu']['sarek'] = load_data_totals('hu', 'sarek', len(rapl_data['hu']['sarek']))

table_data = []
table_head = ['cluster', 'workflow', 'run', 'ichnos (kWh)', 'rapl (kWh)', 'error (%)', 'total_read_bytes', 'total_write_bytes']

for cluster in ['hu', 'gu']:
    for workflow in rapl_data[cluster].keys():
        for run in range(1, len(rapl_data[cluster][workflow].keys()) + 1):
            rapl_entry = rapl_data[cluster][workflow][run]
            ichnos_entry = ichnos_data[cluster][workflow][run]
            bytes_entry = bytes_data[cluster][workflow][run]
            table_data.append([cluster, workflow, run, round(ichnos_entry['total'], 2), round(rapl_entry['total'], 2), get_error(ichnos_entry['total'], rapl_entry['total']), bytes_entry['total_read_bytes'], bytes_entry['total_write_bytes']])

print(tabulate(table_data, table_head))

Cluster Workflow Executions
cluster    workflow      run    ichnos (kWh)    rapl (kWh)    error (%)    total_read_bytes    total_write_bytes
---------  ----------  -----  --------------  ------------  -----------  ------------------  -------------------
hu         rnaseq          1            2.17          2.55        14.93         6.73783e+11          1.45458e+12
hu         rnaseq          2            2.16          2.6         17.1          6.732e+11            1.45454e+12
hu         rnaseq          3            2.17          1.7         27.53         6.6938e+11           1.45483e+12
hu         rnaseq          4            2.27          2.11         7.13         3.32928e+10          1.43058e+12
hu         rnaseq          5            2.27          2.44         6.77         2.16072e+10          1.43057e+12
hu         chipseq         1            3.66          3.85         5.1          2.42034e+11          1.32574e+12
hu         chipseq         2            3.68          3.87         4

In [ ]:
# with open('exported.csv', 'w') as file:
#     file.write('workflow,id,total_read_bytes,total_write_bytes,rapl\n')
#     for row in table_data:
#         if row[0] == 'hu':
#             file.write(f'{row[1]},{row[2]},{row[6]},{row[7]},{row[4]}\n')